# B1 Mapping - Actual Flip Angle (AFI) Imaging

B1 inhomogeneity mapping using the Actual Flip Angle (AFI) method based on a dual-TR FLASH (gradient echo) sequence.
Two complete images are acquired at different repetition times (TR1 and TR2). The ratio of the two images is used to estimate the actual flip angle map, which reflects the B1 field inhomogeneity.

### Imports

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
from einops import rearrange
from mrpro.algorithms.reconstruction import DirectReconstruction
from mrpro.data import KData
from mrpro.data import SpatialDimension
from mrpro.data.traj_calculators import KTrajectoryCartesian

from mrseq.scripts.b1_afi_gre_dual_tr import main as create_seq
from mrseq.utils import sys_defaults

### Settings
We are going to use a numerical phantom with a matrix size of 64 x 64. The AFI sequence uses two different repetition times (TR1=20ms and TR2=100ms) to generate sensitivity to B1 inhomogeneity.

In [ ]:
image_matrix_size = [128, 128]

tmp = tempfile.TemporaryDirectory()
fname_mrd = Path(tmp.name) / 'afi_b1_mapping.mrd'

# AFI sequence parameters
tr1 = 25e-3  # First TR in seconds
tr2 = 200e-3  # Second TR in seconds (5x TR1)
flip_angle = 60  # Nominal flip angle in degrees

### Create the digital phantom

We use the standard Brainweb phantom from [MRzero](https://github.com/MRsources/MRzero-Core) with its default B1 field inhomogeneity.

In [ ]:
phantom = mr0.util.load_phantom(image_matrix_size)
phantom.B0[:] = 0

### Create the AFI FLASH sequence

To create the AFI sequence, we use the previously imported [afi_gre_dual_tr script](../src/mrseq/scripts/afi_gre_dual_tr.py).

In [ ]:
sequence, fname_seq = create_seq(
    system=sys_defaults,
    test_report=False,
    timing_check=False,
    te=None,
    tr1=tr1,
    tr2=tr2,
    fov_xy=float(phantom.size.numpy()[0]),
    n_readout=image_matrix_size[0],
    n_phase_encoding=image_matrix_size[1],
    fov_z=8e-3,
    rf_flip_angle=flip_angle,
)

### Simulate the sequence
Now, we pass the sequence and the phantom to the MRzero simulation and save the simulated signal as an (ISMR)MRD file.

In [ ]:
mr0_sequence = mr0.Sequence.import_file(str(fname_seq.with_suffix('.seq')))
signal, ktraj_adc = mr0.util.simulate(mr0_sequence, phantom, accuracy=1e-3)
mr0.sig_to_mrd(fname_mrd, signal, sequence)

### Reconstruct the image

We use basic FFT reconstruction since we're working with single-coil data from the simulated phantom.

In [ ]:
kdata = KData.from_file(fname_mrd, trajectory=KTrajectoryCartesian())
kdata.header.encoding_matrix = SpatialDimension(z=1, y=image_matrix_size[1], x=image_matrix_size[0])
kdata.header.recon_matrix = SpatialDimension(z=1, y=image_matrix_size[1], x=image_matrix_size[0])
recon = DirectReconstruction(kdata)
idata = recon(kdata)

### Plot images at different TR values

We can now plot the two images acquired at TR1 and TR2. The contrast between tissues is different due to different T1 recovery at the two TR values.

In [ ]:
idat = idata.rss().abs().numpy().squeeze()
fig, ax = plt.subplots(1, 2, figsize=(10, 4))

tr_labels = [f'TR1 = {int(tr1 * 1e3)} ms', f'TR2 = {int(tr2 * 1e3)} ms']

for i in range(2):
    ax[i].imshow(idat[i, :, :], cmap='gray')
    ax[i].set_title(tr_labels[i])
    ax[i].set_xticks([])
    ax[i].set_yticks([])

### Estimate the B1 map

We use the AFI reconstruction formula to estimate the B1 inhomogeneity map from the ratio of the two images.

In [ ]:
def afi_b1_reconstruction(image_tr1, image_tr2, tr1, tr2, nominal_fa_deg):
    """Reconstruct B1 map from dual-TR FLASH (AFI) acquisitions.

    Based on: Yarnykh VL, "Actual flip-angle imaging in the pulsed steady state"
    Magn Reason Med, 57(1):192-200, Jan 2007

    Parameters
    ----------
    image_tr1
        Complex magnitude image acquired at TR1.
    image_tr2
        Complex magnitude image acquired at TR2.
    tr1
        First repetition time in seconds.
    tr2
        Second repetition time in seconds.
    nominal_fa_deg
        Nominal flip angle in degrees.

    Returns
    -------
    b1_map
        B1 inhomogeneity map (actual_FA / nominal_FA).

    """
    # Calculate signal ratio
    s1 = np.abs(image_tr1)
    s2 = np.abs(image_tr2)
    r = s2 / (s1 + 1e-10)

    # AFI formula: N = TR2/TR1
    n = tr2 / tr1

    # Actual flip angle from ratio
    cos_arg = (r * n - 1) / (n - r + 1e-10)
    cos_arg = np.clip(cos_arg, -1.0, 1.0)

    alpha_actual = np.arccos(cos_arg)

    # Normalize by nominal flip angle
    b1_map = alpha_actual / np.deg2rad(nominal_fa_deg)

    return b1_map.astype(np.float32)

In [ ]:
# Reconstruct B1 map using AFI formula
b1_measured = afi_b1_reconstruction(
    image_tr1=idata.data[0].numpy(), image_tr2=idata.data[1].numpy(), tr1=tr1, tr2=tr2, nominal_fa_deg=flip_angle
)

print(f'B1 map reconstructed: min={b1_measured.min():.3f}, max={b1_measured.max():.3f}')
print(f'Mean B1: {b1_measured.mean():.3f}')

We compare the reconstructed B1 map with the input B1 field from the phantom to evaluate the accuracy of the AFI method.

In [ ]:
rb1_input = np.abs(
    np.roll(rearrange(phantom.B1.numpy().squeeze()[::-1, ::-1], 'x y -> y x'), shift=(1, 1), axis=(0, 1))
)
obj_mask = np.zeros_like(rb1_input)
obj_mask[rb1_input > 0] = 1
rb1_measured = np.squeeze(b1_measured * obj_mask)


fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for cax in ax:
    cax.set_xticks([])
    cax.set_yticks([])

im = ax[0].imshow(rb1_input, vmin=0.5, vmax=1.5, cmap='magma')
fig.colorbar(im, ax=ax[0], label='rB1')

im = ax[1].imshow(rb1_measured, vmin=0.5, vmax=1.5, cmap='magma')
fig.colorbar(im, ax=ax[1], label='rB1')

# Difference
im = ax[2].imshow(rb1_measured - rb1_input, cmap='bwr', vmin=-0.5, vmax=0.5)
fig.colorbar(im, ax=ax[2], label='Difference')
plt.tight_layout()

relative_error = np.sum(np.abs(rb1_input - rb1_measured)) / np.sum(np.abs(rb1_input))
print(f'Relative error {relative_error}')
assert relative_error < 0.10